In [1]:
# ============================================================
# E03 — SHAP ANALYSIS FOR POOLED RANDOM FOREST
# ============================================================
#
# Purpose:
# Explain the pooled Random Forest from E03 using SHAP.
#
# The model is trained on:
#   Winter train
#   Summer train
#   Monsoon train
#   Post-monsoon train
#       ↓
#   Pooled training data
#       ↓
#   StandardScaler
#       ↓
#   SMOTE
#       ↓
#   ONE pooled Random Forest
#
# SHAP is then calculated separately on:
#   Winter test
#   Summer test
#   Monsoon test
#   Post-monsoon test
#
# ============================================================


# ============================================================
# STEP 0: Install and import packages
# ============================================================

!pip install -q shap imbalanced-learn

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

print("Packages imported successfully.")


# ============================================================
# STEP 1: Configuration
# ============================================================

UPLOAD_DIR = "/content"

# Output directory
OUTPUT_DIR = os.path.join(
    "results",
    "shap",
    "e03_pooled_rf"
)

PLOTS_DIR = os.path.join(
    OUTPUT_DIR,
    "plots"
)

TABLES_DIR = os.path.join(
    OUTPUT_DIR,
    "tables"
)

os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)


# ============================================================
# STEP 2: Predictors, target and seasons
# EXACTLY SAME AS E03
# ============================================================

predictors = [
    'AQI_lag24',
    'Raw_Conc_lag24',
    'Hour',
    'Month',
    'DayOfWeek',
    'temperature_2m',
    'relativehumidity_2m',
    'dewpoint_2m',
    'apparent_temperature',
    'precipitation',
    'rain',
    'surface_pressure',
    'cloudcover',
    'cloudcover_low',
    'cloudcover_mid',
    'cloudcover_high',
    'windspeed_10m',
    'winddirection_10m',
    'windgusts_10m',
    'et0_fao_evapotranspiration',
    'vapor_pressure_deficit',
    'shortwave_radiation',
    'direct_radiation',
    'diffuse_radiation',
    'weathercode'
]

target = 'Hazard_Tier'

seasons = [
    'winter',
    'summer',
    'monsoon',
    'post_monsoon'
]


# ============================================================
# STEP 3: Check that required CSV files exist
# ============================================================

print("\n" + "=" * 70)
print("CHECKING INPUT FILES")
print("=" * 70)

for season in seasons:

    possible_names = [
        f"{season}.csv",
        f"data_{season}.csv",
        f"dhaka_{season}.csv"
    ]

    found = False

    for name in possible_names:

        file_path = os.path.join(
            UPLOAD_DIR,
            name
        )

        if os.path.exists(file_path):

            print(f"[OK] {name}")
            found = True
            break

    if not found:

        raise FileNotFoundError(
            f"\nCould not find CSV for {season}.\n"
            f"Upload one of these files to /content:\n"
            f"{possible_names}"
        )


# ============================================================
# STEP 4: Load seasonal data
# Apply EXACT SAME 80/20 time-based split as E03
# ============================================================

print("\n" + "=" * 70)
print("LOADING SEASONAL DATA")
print("=" * 70)

raw_train_frames_X = []
raw_train_frames_y = []

season_test_sets = {}


for season in seasons:

    possible_names = [
        f"{season}.csv",
        f"data_{season}.csv",
        f"dhaka_{season}.csv"
    ]

    file_path = None

    for name in possible_names:

        candidate = os.path.join(
            UPLOAD_DIR,
            name
        )

        if os.path.exists(candidate):

            file_path = candidate
            break

    if file_path is None:

        raise FileNotFoundError(
            f"Could not find CSV for {season}."
        )


    # Load CSV
    df = pd.read_csv(file_path)


    # Check required columns
    missing_columns = [
        col for col in predictors + [target]
        if col not in df.columns
    ]

    if missing_columns:

        raise ValueError(
            f"\n{season}: Missing columns:\n"
            f"{missing_columns}"
        )


    # Check NaNs
    nan_count = df[predictors].isna().sum().sum()

    if nan_count > 0:

        raise ValueError(
            f"\n{season}: Found {nan_count} NaN values "
            f"in predictor columns."
        )


    # ========================================================
    # EXACT E03 TIME-BASED 80/20 SPLIT
    # ========================================================

    split_idx = int(len(df) * 0.8)

    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]


    # Training data
    raw_train_frames_X.append(
        train_df[predictors]
    )

    raw_train_frames_y.append(
        train_df[target]
    )


    # Test data
    season_test_sets[season] = {

        'X_test_raw':
            test_df[predictors].copy(),

        'y_test_raw':
            test_df[target].copy()
    }


    print(
        f"{season.capitalize():<15} "
        f"Total={len(df):>6} | "
        f"Train={len(train_df):>6} | "
        f"Test={len(test_df):>6}"
    )


# ============================================================
# STEP 5: Create pooled training dataset
# EXACT SAME AS E03
# ============================================================

print("\n" + "=" * 70)
print("CREATING POOLED TRAINING DATA")
print("=" * 70)


X_pool_raw = pd.concat(
    raw_train_frames_X,
    axis=0
).reset_index(drop=True)


y_pool_raw = pd.concat(
    raw_train_frames_y,
    axis=0
).reset_index(drop=True)


print(
    f"Pooled training samples: "
    f"{X_pool_raw.shape[0]}"
)

print(
    f"Number of features: "
    f"{X_pool_raw.shape[1]}"
)

print("\nPooled class distribution:")
print(y_pool_raw.value_counts())


# ============================================================
# STEP 6: Fit StandardScaler
# EXACT SAME AS E03
# ============================================================

print("\n" + "=" * 70)
print("FITTING POOLED STANDARD SCALER")
print("=" * 70)


pooled_scaler = StandardScaler()


X_pool_scaled = pooled_scaler.fit_transform(
    X_pool_raw
)


# Scale each season's test set
for season in seasons:

    X_test_raw = (
        season_test_sets[season]
        ['X_test_raw']
    )

    X_test_scaled = pooled_scaler.transform(
        X_test_raw
    )

    season_test_sets[season][
        'X_test_scaled'
    ] = X_test_scaled


print("Scaler fitted successfully.")


# ============================================================
# STEP 7: Apply SMOTE
# EXACT SAME AS E03
# ============================================================

print("\n" + "=" * 70)
print("APPLYING SMOTE")
print("=" * 70)


smote = SMOTE(
    random_state=42
)


X_pool_res, y_pool_res = (
    smote.fit_resample(
        X_pool_scaled,
        y_pool_raw
    )
)


print(
    f"Before SMOTE: "
    f"{X_pool_scaled.shape[0]}"
)

print(
    f"After SMOTE: "
    f"{X_pool_res.shape[0]}"
)

print("\nPost-SMOTE class distribution:")
print(
    pd.Series(y_pool_res).value_counts()
)


# ============================================================
# STEP 8: Train the POOLED RANDOM FOREST
# EXACT SAME HYPERPARAMETERS AS E03
# ============================================================

print("\n" + "=" * 70)
print("TRAINING POOLED RANDOM FOREST")
print("=" * 70)


pooled_rf = RandomForestClassifier(

    n_estimators=100,

    max_depth=15,

    class_weight='balanced',

    random_state=42,

    n_jobs=-1
)


pooled_rf.fit(
    X_pool_res,
    y_pool_res
)


print(
    f"Random Forest trained successfully."
)

print(
    f"Number of trees: "
    f"{pooled_rf.n_estimators}"
)

print(
    f"Classes: "
    f"{pooled_rf.classes_}"
)


# ============================================================
# STEP 9: Create SHAP TreeExplainer
# ============================================================

print("\n" + "=" * 70)
print("CREATING SHAP EXPLAINER")
print("=" * 70)


explainer = shap.TreeExplainer(
    pooled_rf
)


print("SHAP TreeExplainer created.")


# ============================================================
# STEP 10: Calculate SHAP values for each season
# ============================================================

print("\n" + "=" * 70)
print("CALCULATING SHAP VALUES")
print("=" * 70)


shap_values_seasons = {}


for season in seasons:

    print(
        f"\nCalculating SHAP for "
        f"{season.capitalize()}..."
    )


    # Exact E03 test data
    X_test_scaled = (
        season_test_sets[season]
        ['X_test_scaled']
    )


    # Convert to DataFrame to preserve feature names
    X_test_df = pd.DataFrame(
        X_test_scaled,
        columns=predictors
    )


    # Calculate SHAP
    shap_values = explainer.shap_values(
        X_test_df
    )


    # Save
    shap_values_seasons[season] = {

        'X_test':
            X_test_df,

        'shap_values':
            shap_values
    }


    print(
        f"Completed {season.capitalize()} "
        f"({len(X_test_df)} samples)"
    )


# ============================================================
# STEP 11: Display SHAP output structure
# ============================================================

print("\n" + "=" * 70)
print("SHAP OUTPUT STRUCTURE")
print("=" * 70)


for season in seasons:

    shap_values = (
        shap_values_seasons[season]
        ['shap_values']
    )

    print(
        f"\n{season.capitalize()}:"
    )

    print(
        "Type:",
        type(shap_values)
    )


    if isinstance(
        shap_values,
        list
    ):

        print(
            "Number of classes:",
            len(shap_values)
        )

        for class_idx, values in enumerate(
            shap_values
        ):

            print(
                f"  Class {pooled_rf.classes_[class_idx]}: "
                f"{values.shape}"
            )

    else:

        print(
            "Shape:",
            shap_values.shape
        )


# ============================================================
# STEP 12: Create SHAP summary plots
# ============================================================

print("\n" + "=" * 70)
print("GENERATING SHAP SUMMARY PLOTS")
print("=" * 70)


for season in seasons:

    print(
        f"\nGenerating plots for "
        f"{season.capitalize()}..."
    )


    X_test_df = (
        shap_values_seasons[season]
        ['X_test']
    )

    shap_values = (
        shap_values_seasons[season]
        ['shap_values']
    )


    # --------------------------------------------------------
    # SHAP version where multiclass output is a LIST
    # --------------------------------------------------------

    if isinstance(
        shap_values,
        list
    ):

        for class_idx in range(
            len(shap_values)
        ):

            class_value = (
                pooled_rf.classes_[class_idx]
            )


            plt.figure(
                figsize=(10, 7)
            )


            shap.summary_plot(

                shap_values[class_idx],

                X_test_df,

                show=False
            )


            plt.title(
                f"SHAP Summary — "
                f"{season.capitalize()} — "
                f"Class {class_value}"
            )


            plt.tight_layout()


            filename = (
                f"shap_summary_"
                f"{season}_"
                f"class_{class_value}.png"
            )


            save_path = os.path.join(
                PLOTS_DIR,
                filename
            )


            plt.savefig(
                save_path,
                dpi=300,
                bbox_inches='tight'
            )


            plt.show()

            plt.close()


            print(
                f"Saved: {save_path}"
            )


    # --------------------------------------------------------
    # SHAP version where output is 3D:
    # samples × features × classes
    # --------------------------------------------------------

    elif len(shap_values.shape) == 3:

        n_classes = (
            shap_values.shape[2]
        )


        for class_idx in range(
            n_classes
        ):

            class_value = (
                pooled_rf.classes_[class_idx]
            )


            plt.figure(
                figsize=(10, 7)
            )


            shap.summary_plot(

                shap_values[:, :, class_idx],

                X_test_df,

                show=False
            )


            plt.title(
                f"SHAP Summary — "
                f"{season.capitalize()} — "
                f"Class {class_value}"
            )


            plt.tight_layout()


            filename = (
                f"shap_summary_"
                f"{season}_"
                f"class_{class_value}.png"
            )


            save_path = os.path.join(
                PLOTS_DIR,
                filename
            )


            plt.savefig(
                save_path,
                dpi=300,
                bbox_inches='tight'
            )


            plt.show()

            plt.close()


            print(
                f"Saved: {save_path}"
            )


# ============================================================
# STEP 13: Calculate Mean Absolute SHAP Importance
# ============================================================

print("\n" + "=" * 70)
print("CALCULATING MEAN |SHAP| IMPORTANCE")
print("=" * 70)


importance_tables = {}


for season in seasons:

    shap_values = (
        shap_values_seasons[season]
        ['shap_values']
    )


    season_tables = []


    # --------------------------------------------------------
    # LIST FORMAT
    # --------------------------------------------------------

    if isinstance(
        shap_values,
        list
    ):

        for class_idx, class_shap in enumerate(
            shap_values
        ):

            mean_abs_shap = (
                np.abs(class_shap)
                .mean(axis=0)
            )


            importance_df = pd.DataFrame({

                'Feature':
                    predictors,

                'Mean_Abs_SHAP':
                    mean_abs_shap
            })


            importance_df[
                'Class'
            ] = pooled_rf.classes_[class_idx]


            importance_df = (
                importance_df
                .sort_values(
                    'Mean_Abs_SHAP',
                    ascending=False
                )
                .reset_index(drop=True)
            )


            importance_df[
                'Rank'
            ] = np.arange(
                1,
                len(importance_df) + 1
            )


            season_tables.append(
                importance_df
            )


    # --------------------------------------------------------
    # 3D FORMAT
    # --------------------------------------------------------

    elif len(shap_values.shape) == 3:

        n_classes = (
            shap_values.shape[2]
        )


        for class_idx in range(
            n_classes
        ):

            class_shap = (
                shap_values[:, :, class_idx]
            )


            mean_abs_shap = (
                np.abs(class_shap)
                .mean(axis=0)
            )


            importance_df = pd.DataFrame({

                'Feature':
                    predictors,

                'Mean_Abs_SHAP':
                    mean_abs_shap
            })


            importance_df[
                'Class'
            ] = pooled_rf.classes_[class_idx]


            importance_df = (
                importance_df
                .sort_values(
                    'Mean_Abs_SHAP',
                    ascending=False
                )
                .reset_index(drop=True)
            )


            importance_df[
                'Rank'
            ] = np.arange(
                1,
                len(importance_df) + 1
            )


            season_tables.append(
                importance_df
            )


    importance_tables[season] = (
        pd.concat(
            season_tables,
            ignore_index=True
        )
    )


# ============================================================
# STEP 14: Save SHAP importance tables
# ============================================================

print("\n" + "=" * 70)
print("SAVING SHAP TABLES")
print("=" * 70)


for season in seasons:

    df_importance = (
        importance_tables[season]
    )


    output_path = os.path.join(

        TABLES_DIR,

        f"shap_importance_"
        f"{season}.csv"
    )


    df_importance.to_csv(
        output_path,
        index=False
    )


    print(
        f"Saved: {output_path}"
    )


# ============================================================
# STEP 15: Create overall feature ranking per season
# ============================================================
#
# This averages Mean |SHAP| across all classes.
#
# Useful for answering:
# "Which features are most important overall
#  for this season?"
#
# ============================================================

overall_importance = {}


for season in seasons:

    df = importance_tables[season]


    overall = (
        df.groupby('Feature')
        ['Mean_Abs_SHAP']
        .mean()
        .sort_values(
            ascending=False
        )
        .reset_index()
    )


    overall['Rank'] = np.arange(
        1,
        len(overall) + 1
    )


    overall_importance[season] = (
        overall
    )


    output_path = os.path.join(

        TABLES_DIR,

        f"overall_shap_importance_"
        f"{season}.csv"
    )


    overall.to_csv(
        output_path,
        index=False
    )


    print(
        f"Saved: {output_path}"
    )


# ============================================================
# STEP 16: Print top 10 features for each season
# ============================================================

print("\n" + "=" * 70)
print("TOP 10 FEATURES — MEAN |SHAP|")
print("=" * 70)


for season in seasons:

    print(
        f"\n{'=' * 50}"
    )

    print(
        f"{season.upper()}"
    )

    print(
        f"{'=' * 50}"
    )


    print(
        overall_importance[season]
        .head(10)
        .to_string(index=False)
    )


# ============================================================
# STEP 17: Create comparison table
# ============================================================
#
# Shows the overall Mean |SHAP| importance of each feature
# across all four seasons.
#
# ============================================================

comparison_df = pd.DataFrame()


for season in seasons:

    temp = (
        overall_importance[season]
        [['Feature', 'Mean_Abs_SHAP']]
        .rename(
            columns={
                'Mean_Abs_SHAP':
                    season
            }
        )
    )


    if comparison_df.empty:

        comparison_df = temp

    else:

        comparison_df = comparison_df.merge(
            temp,
            on='Feature',
            how='outer'
        )


comparison_path = os.path.join(
    TABLES_DIR,
    "shap_feature_importance_comparison.csv"
)


comparison_df.to_csv(
    comparison_path,
    index=False
)


print(
    "\nSaved comparison table:"
)

print(
    comparison_path
)


# ============================================================
# STEP 18: Create one comparison plot
# ============================================================
#
# Top 10 features based on average importance across seasons.
#
# ============================================================

comparison_df['Average'] = (
    comparison_df[
        seasons
    ].mean(axis=1)
)


top_features = (
    comparison_df
    .sort_values(
        'Average',
        ascending=False
    )
    .head(10)
)


plt.figure(
    figsize=(11, 7)
)


plt.barh(
    top_features['Feature'][::-1],
    top_features['Average'][::-1]
)


plt.xlabel(
    'Mean Absolute SHAP Value'
)

plt.ylabel(
    'Feature'
)

plt.title(
    'Top 10 Features — Pooled Random Forest SHAP'
)

plt.tight_layout()


comparison_plot_path = os.path.join(
    PLOTS_DIR,
    "top10_feature_importance_across_seasons.png"
)


plt.savefig(
    comparison_plot_path,
    dpi=300,
    bbox_inches='tight'
)


plt.show()

plt.close()


print(
    f"\nSaved: {comparison_plot_path}"
)


# ============================================================
# STEP 19: Final output summary
# ============================================================

print("\n")
print("=" * 70)
print("SHAP ANALYSIS COMPLETED SUCCESSFULLY")
print("=" * 70)

print("\nOutput directory:")
print(
    os.path.abspath(OUTPUT_DIR)
)

print("\nPlots:")
print(
    os.path.abspath(PLOTS_DIR)
)

print("\nTables:")
print(
    os.path.abspath(TABLES_DIR)
)

print("\nFiles generated:")

for root, dirs, files in os.walk(
    OUTPUT_DIR
):

    for file in files:

        print(
            os.path.join(
                root,
                file
            )
        )

Output hidden; open in https://colab.research.google.com to view.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import shutil
shutil.copytree(
    '/content/results/shap/e03_pooled_rf/tables',
    '/content/drive/MyDrive/AQI_Seasonal_Prediction/results/shap/tables',
    dirs_exist_ok=True
)

'/content/drive/MyDrive/AQI_Seasonal_Prediction/results/shap/tables'